# SO-101 cube pick-and-place: randomised re-render + SmolVLA training (Colab)

What this notebook does, top to bottom:

1. **Setup**: mounts Drive, installs LeRobot (the exact version used locally and on Iridis) and MuJoCo, and sets up headless GPU rendering.
2. **Preview**: renders 12 randomised frames so you can check the look before spending hours.
3. **Render**: re-renders the 148 demos × `COPIES` with randomised visuals (lighting, soft shadows, materials, camera jitter, stray objects, blur/noise/JPEG), in parallel, then merges them into one LeRobot dataset. ~2–3 h. The result is saved to Drive as a tar, so a new session restores it in minutes instead of re-rendering.
4. **Train**: fine-tunes `lerobot/smolvla_base`, with the same settings as the Iridis run that got 93% in sim, plus LeRobot's image augmentation. Checkpoints are copied to Drive as they are written, and re-running the cell **resumes** after a disconnect.
5. **Sim check**: evaluates the trained policy on the 30 sim test poses.

**Before you start**
- Runtime → Change runtime type → **A100 GPU** (High-RAM if offered).
- Upload `vla_bundle.zip` (made on the Mac by `sim/colab/make_bundle.py`) to `MyDrive/so101_vla/`, or change `DRIVE_DIR` below.
- Keep the tab open, or use background execution. If the session drops, run cells 1, 2 and 4 again, then the cell you were on. Rendering restores from Drive and training resumes from the last checkpoint.

In [ ]:
# 1. Settings
DRIVE_DIR = "/content/drive/MyDrive/so101_vla"   # holds vla_bundle.zip; dataset tar + checkpoints go here too
BUNDLE = "vla_bundle.zip"
DATASET = "so101-cube-dr"                        # name of the randomised dataset
COPIES = 4                                       # randomised copies of each of the 148 demos
SEED = 0
RUN = "smolvla_cube_dr"                          # training run name
STEPS = 30000
BATCH = 64
SAVE_EVERY = 2500
KEEP_DATASET_ON_DRIVE = True                     # ~8 GB tar on Drive: a new session skips the render
LEROBOT_COMMIT = "b9cb121cb7d4e3e26ec5c906d08dda68d151ba52"   # = the local / Iridis install (0.6.2)
WORK = "/content/vla"                            # the unpacked bundle (the coding/ folder tree)
SIM = f"{WORK}/sim"
SCENE = "mujoco_menagerie/robotstudio_so101/pick_place_scene_real.xml"

In [ ]:
# 2. Drive and hardware
import os, shutil
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print("CPUs:", os.cpu_count(), "| free disk:", round(shutil.disk_usage("/content").free / 1e9), "GB")
assert os.path.exists(f"{DRIVE_DIR}/{BUNDLE}"), f"upload {BUNDLE} to {DRIVE_DIR} first"

In [ ]:
# 3. Install (a few minutes; once per session)
!pip install -q "lerobot[smolvla,training] @ git+https://github.com/huggingface/lerobot.git@{LEROBOT_COMMIT}" "mujoco==3.16.0"
!python -c "import lerobot, mujoco, torch; print('lerobot', lerobot.__version__, '| mujoco', mujoco.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"
# If the line above fails with an import error: Runtime -> Restart session, run cells 1 and 2, skip this one.

In [ ]:
# 4. Headless rendering (EGL on the GPU) and the bundle
import os, zipfile
icd = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(icd):
    os.makedirs(os.path.dirname(icd), exist_ok=True)
    with open(icd, "w") as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}')
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
if not os.path.exists(f"{SIM}/randomise_render.py"):
    zipfile.ZipFile(f"{DRIVE_DIR}/{BUNDLE}").extractall(WORK)
%cd {SIM}
!python -c "import mujoco; m = mujoco.MjModel.from_xml_path('{SCENE}'); d = mujoco.MjData(m); mujoco.mj_forward(m, d); r = mujoco.Renderer(m, 48, 64); r.update_scene(d); print('EGL render OK, mean pixel', round(float(r.render().mean()), 1))"

# video codec for the dataset: H.264 like the original dataset if this ffmpeg has it, else LeRobot's choice
import av
def has_encoder(name):
    try:
        av.codec.Codec(name, "w")
        return True
    except Exception:
        return False
VCODEC = "h264" if has_encoder("libx264") else "auto"
print("video codec:", VCODEC)

In [ ]:
# 5. Preview: 12 randomised frames (side | wrist). Check the look before rendering everything.
!python randomise_render.py --preview 12 --pairs "" --scene {SCENE}
from IPython.display import Image, display
display(Image(f"{SIM}/eval/dr_preview.png"))

In [ ]:
# 6. Render the randomised dataset in parallel shards, merge, save to Drive (~2-3 h; skipped if done)
import glob, json, subprocess, time
ROOT = f"{SIM}/lerobot_dataset"
FINAL = f"{ROOT}/{DATASET}"
TAR = f"{DRIVE_DIR}/{DATASET}.tar"
SHARDS = f"{ROOT}/{DATASET}_shards"

if os.path.exists(f"{FINAL}/meta/info.json"):
    print("dataset already here")
elif os.path.exists(TAR):
    print("restoring the dataset from Drive ...")
    os.makedirs(ROOT, exist_ok=True)
    !tar -xf "{TAR}" -C "{ROOT}"
else:
    N = max(1, min(os.cpu_count() - 1, 12))
    print(f"rendering {148 * COPIES} episodes in {N} shards")
    procs = []
    for i in range(N):
        log = open(f"/content/render_{i}.log", "w")
        procs.append(subprocess.Popen(
            ["python", "randomise_render.py", "demos/cube/trimmed/*.npz", "--copies", str(COPIES),
             "--seed", str(SEED), "--scene", SCENE, "--vcodec", VCODEC, "--shard", f"{i}/{N}",
             "--repo-id", f"local/{DATASET}_shard{i}", "--root", f"lerobot_dataset/{DATASET}_shards/shard{i}",
             "--overwrite"], stdout=log, stderr=subprocess.STDOUT))
    t0 = time.time()
    while any(p.poll() is None for p in procs):
        time.sleep(120)
        left = []
        for i in range(N):
            lines = [l for l in open(f"/content/render_{i}.log") if "min left" in l]
            left.append(lines[-1].split("~")[-1].split(" min")[0] if lines else "?")
        print(f"{(time.time() - t0) / 60:.0f} min: minutes left per shard {left}", flush=True)
    bad = [i for i, p in enumerate(procs) if p.returncode != 0]
    assert not bad, f"shard(s) {bad} failed: see /content/render_<i>.log"

    ids = "[" + ",".join(f"'local/{DATASET}_shard{i}'" for i in range(N)) + "]"
    roots = "[" + ",".join(f"'{SHARDS}/shard{i}'" for i in range(N)) + "]"
    subprocess.run(["lerobot-edit-dataset", "--new_repo_id", f"local/{DATASET}", "--new_root", FINAL,
                    "--operation.type", "merge", "--operation.repo_ids", ids, "--operation.roots", roots], check=True)
    with open(f"{FINAL}/meta/randomisation_shards.jsonl", "w") as out:   # every episode's random settings
        for i in range(N):
            for line in open(f"{SHARDS}/shard{i}/meta/randomisation.jsonl"):
                rec = json.loads(line)
                rec["shard"] = i
                out.write(json.dumps(rec) + "\n")
    shutil.copy(f"{SHARDS}/shard0/joint_map.json", FINAL)
    shutil.rmtree(SHARDS)
    if KEEP_DATASET_ON_DRIVE:
        print("saving the dataset to Drive ...")
        !tar -cf "{TAR}" -C "{ROOT}" "{DATASET}"

info = json.load(open(f"{FINAL}/meta/info.json"))
print(f"{DATASET}: {info['total_episodes']} episodes, {info['total_frames']} frames "
      f"(expected {148 * COPIES} episodes, {72966 * COPIES} frames)")

In [ ]:
# 7. Train SmolVLA (resumes from the last checkpoint on Drive if there is one)
import glob, subprocess, time
OUT = f"/content/outputs/train/{RUN}"
CK = f"{OUT}/checkpoints"
DRIVE_CK = f"{DRIVE_DIR}/checkpoints/{RUN}"
os.makedirs(DRIVE_CK, exist_ok=True)

def steps_in(d):
    return sorted(x for x in os.listdir(d) if x.isdigit()) if os.path.isdir(d) else []

# a new session: bring the newest checkpoint back from Drive
if not os.path.exists(f"{CK}/last") and steps_in(DRIVE_CK):
    s = steps_in(DRIVE_CK)[-1]
    shutil.copytree(f"{DRIVE_CK}/{s}", f"{CK}/{s}")
    os.symlink(s, f"{CK}/last")
    print("restored checkpoint", s, "from Drive")

def sync(keep=2):
    # copy the newest COMPLETE checkpoint (the one 'last' points to) to Drive; keep `keep` there and here
    if not os.path.islink(f"{CK}/last"):
        return
    s = os.readlink(f"{CK}/last").rstrip("/").split("/")[-1]
    if not os.path.exists(f"{DRIVE_CK}/{s}"):
        shutil.copytree(f"{CK}/{s}", f"{DRIVE_CK}/{s}.tmp")
        os.rename(f"{DRIVE_CK}/{s}.tmp", f"{DRIVE_CK}/{s}")
        print("  checkpoint", s, "-> Drive", flush=True)
    for d in (DRIVE_CK, CK):
        for old in steps_in(d)[:-keep]:
            shutil.rmtree(f"{d}/{old}", ignore_errors=True)

cfg = f"{CK}/last/pretrained_model/train_config.json"
if os.path.exists(cfg):
    cmd = ["lerobot-train", f"--config_path={cfg}", "--resume=true"]
else:
    shutil.rmtree(OUT, ignore_errors=True)
    cmd = ["lerobot-train",
           "--policy.path=lerobot/smolvla_base",
           f"--dataset.repo_id=local/{DATASET}", f"--dataset.root={FINAL}",
           '--rename_map={"observation.images.ext_cam": "observation.images.camera1", '
           '"observation.images.wrist_cam": "observation.images.camera2"}',
           "--dataset.image_transforms.enable=true",
           "--policy.device=cuda", "--policy.push_to_hub=false",
           f"--policy.scheduler_decay_steps={STEPS}",
           f"--output_dir={OUT}", f"--job_name={RUN}",
           f"--batch_size={BATCH}", f"--steps={STEPS}", f"--save_freq={SAVE_EVERY}", "--log_freq=100",
           f"--num_workers={max(1, os.cpu_count() - 1)}", "--wandb.enable=false"]
print(" ".join(cmd))
log_path = f"/content/train_{RUN}.log"
proc = subprocess.Popen(cmd, stdout=open(log_path, "a"), stderr=subprocess.STDOUT)
while proc.poll() is None:
    time.sleep(120)
    steps = [l.strip() for l in open(log_path) if "step:" in l]
    print(steps[-1] if steps else "(starting ...)", flush=True)
    sync()
sync()
print("lerobot-train exit code", proc.returncode, "| log:", log_path)
!tail -n 5 {log_path}

In [ ]:
# 8. Sim check: the trained policy on the 30 sim test poses (unrandomised scene), ~10-20 min
# (works on its own after cells 1, 2 and 4: takes the newest checkpoint from Drive if this session has none)
CK = f"/content/outputs/train/{RUN}/checkpoints"
DRIVE_CK = f"{DRIVE_DIR}/checkpoints/{RUN}"
if not os.path.islink(f"{CK}/last"):
    s = sorted(x for x in os.listdir(DRIVE_CK) if x.isdigit())[-1]
    if not os.path.exists(f"{CK}/{s}"):
        shutil.copytree(f"{DRIVE_CK}/{s}", f"{CK}/{s}")
    os.symlink(s, f"{CK}/last")
    print("using checkpoint", s, "from Drive")
CKPT = f"{CK}/last/pretrained_model"
step = os.readlink(f"{CK}/last").rstrip("/").split("/")[-1]
res = f"/content/eval_{RUN}_{step}_test.json"
!python eval_policy.py "{CKPT}" --poses eval/cube_poses.json --split test --out "{res}" --video-dir "/content/eval_videos_{RUN}" --videos 5 2>&1 | grep -v Warning | tail -n 15
shutil.copy(res, DRIVE_DIR)
!cp -r "/content/eval_videos_{RUN}" "{DRIVE_DIR}/"
print("results and videos copied to", DRIVE_DIR)

## Back on the Mac

Download `MyDrive/so101_vla/checkpoints/smolvla_cube_dr/<step>/pretrained_model/` into
`coding/sim/outputs/train/smolvla_cube_dr/<step>/pretrained_model/`, then test on the real arm, as before:

```
cd sim
/opt/anaconda3/envs/lerobot/bin/python run_policy_real.py outputs/train/smolvla_cube_dr/<step>/pretrained_model --split test --step
```